In [ ]:
from pathlib import Path
import sys

current_dir = Path.cwd()

if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate project root containing 'src'."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


import joblib
import numpy as np
import pandas as pd

from keras.models import load_model
from src.evaluation import (
    calculate_metrics,
    plot_predictions
)

model = load_model(
    PROJECT_ROOT / "models/best_eth_lstm.keras"
)

target_scaler = joblib.load(
    PROJECT_ROOT / "models/target_scaler.pkl"
)

In [ ]:
DATA_DIR = Path("../data/processed")

x_test = np.load(DATA_DIR / "x_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
scaled_predictions = model.predict(x_test)

predictions = target_scaler.inverse_transform(
    scaled_predictions
)

actual_values = target_scaler.inverse_transform(
    y_test
)

In [ ]:
target_names = [
    "Open",
    "High",
    "Low",
    "Close"
]

evaluation_results = []

for index, target in enumerate(target_names):
    metrics = calculate_metrics(
        actual_values[:, index],
        predictions[:, index]
    )

    metrics["Target"] = target

    evaluation_results.append(metrics)

metrics_df = pd.DataFrame(evaluation_results)

metrics_df

In [ ]:
metrics_df.to_csv(
    PROJECT_ROOT / "results/metrics.csv",
    index=False
)

In [ ]:
predictions_df = pd.DataFrame({
    "Actual_Open": actual_values[:, 0],
    "Predicted_Open": predictions[:, 0],
    "Actual_High": actual_values[:, 1],
    "Predicted_High": predictions[:, 1],
    "Actual_Low": actual_values[:, 2],
    "Predicted_Low": predictions[:, 2],
    "Actual_Close": actual_values[:, 3],
    "Predicted_Close": predictions[:, 3]
})

predictions_df.to_csv(
    PROJECT_ROOT / "results/predictions.csv",
    index=False
)